# Đề tài 08 - Ordered Treasure Hunt

**Học phần:** Học tăng cường - Reinforcement Learning
**Sinh viên:** Nguyễn Duy Hưng - MSSV 24103225 - Lớp EEE.AI
**Môi trường:** custom `gymnasium.Env` `OrderedTreasureHuntEnv`

Agent phải thu thập các vật phẩm theo một thứ tự hợp lệ rồi mới thoát ra qua ô `X`.
Chuỗi vật phẩm bắt buộc là `A -> B -> C`; ở các setting khác chuỗi có thể có nhiều
nhánh, có vật phẩm bị khóa, và có ô thưởng bỏ qua được một phần tử của chuỗi.

Notebook này chứa **toàn bộ mã nguồn** của project (những file `.py` được chèn trực
tiếp vào các cell code) và chạy lại toàn bộ thí nghiệm.

## 0. Chuẩn bị môi trường

In [3]:
import os
import sys

def find_root(start):
    """Find the project root directory containing envs.py."""
    path = os.path.abspath(start)
    while True:
        if os.path.exists(os.path.join(path, "envs.py")):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise RuntimeError("Không tìm thấy thư mục gốc của project")
        path = parent

ROOT = find_root(os.getcwd())
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("Thư mục gốc :", ROOT)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Matplotlib  :", matplotlib.__version__)
print("Pandas      :", pd.__version__)

Thư mục gốc : e:\python\reinforcement learning\cuoiky
Python      : 3.14.0
NumPy       : 2.3.4
Matplotlib  : 3.11.1
Pandas      : 2.3.3



## 1. Mô hình MDP

| Thành phần | Mô tả |
|---|---|
| State | `(row, col, task_progress, flags)` - vị trí agent, số phần tử đã giải, bitmask biến phụ |
| Action | `0 = up`, `1 = down`, `2 = left`, `3 = right` (4 hành động) |
| Reward | `+10` lấy đúng vật phẩm tiếp theo, `-20` chạm vật phẩm sai thứ tự, `-1` mỗi bước, `+100` thoát qua `X` |
| Terminal | Agent bước vào `X` khi `task_progress` đã đạt giá trị lớn nhất |
| Truncation | Episode dừng sau `max_steps = 100` bước |

**Vì sao `task_progress` phải thuộc về state?** Cùng một ô (2, 1) của level 1 nhưng với
`task_progress = 0` thì bước sang ô của vật phẩm `A` cho `+10`, còn với
`task_progress = 1` thì vật phẩm `A` đã bị lấy và ô đó đã thành sàn, bước vào chỉ
còn cho `-1`. Hai tình huống khác nhau, do đó phải là hai state khác nhau. Phần 7 sẽ
kiểm chứng bằng số liệu.

**Các biến phụ để giữ tính Markov**

- `switch_on` (level 3): ô chứa vật phẩm `C` chỉ vào được khi công tác đã bật.
- `skip_used` (level 3): ô thưởng chỉ bỏ qua được đúng một lần.
- Bitmask của các vật phẩm đã giải (level 2): khi chuỗi có hai nhánh, hai chuỗi khác
  nhau cùng số phần tử đã giải.
  python main.py --autoplay --algo qlearning --config level1 --gif figures/autoplay_level1.gif


## 2. Mã nguồn môi trường (`envs.py`)

In [ ]:

from __future__ import annotations

from collections import deque

import gymnasium as gym
import numpy as np
from gymnasium import spaces


# ============================================================
# ACTION CONSTANTS
# ============================================================

ACTION_UP = 0
ACTION_DOWN = 1
ACTION_LEFT = 2
ACTION_RIGHT = 3

#: Human readable name of every action.
ACTION_NAMES = {
    ACTION_UP: "up",
    ACTION_DOWN: "down",
    ACTION_LEFT: "left",
    ACTION_RIGHT: "right",
}

#: Arrow glyph of every action, used when a policy grid is printed or drawn.
ACTION_SYMBOLS = {
    ACTION_UP: "^",
    ACTION_DOWN: "v",
    ACTION_LEFT: "<",
    ACTION_RIGHT: ">",
}

#: (row, col) displacement produced by every action.
ACTION_DELTAS = {
    ACTION_UP: (-1, 0),
    ACTION_DOWN: (1, 0),
    ACTION_LEFT: (0, -1),
    ACTION_RIGHT: (0, 1),
}

N_ACTIONS = 4


# ============================================================
# TILE LEGEND
# ============================================================

TILE_WALL = "#"
TILE_FLOOR = "."
TILE_START = "S"
TILE_EXIT = "X"
TILE_SWITCH = "W"
TILE_SKIP = "K"


# ============================================================
# DEFAULT REWARDS
# ============================================================

#: Default reward table. Every value can be overridden per level.
DEFAULT_REWARDS = {
    "step_penalty": -1.0,   
    "wall_penalty": -3.0,        
    "wrong_item_penalty": -20.0, 
    "locked_penalty": -8.0,   
    "early_exit_penalty": -20.0, 
    "item_reward": 10.0,     
    "switch_reward": 0.0,    
    "skip_reward": 0.0,         
    "exit_bonus": 100.0,         
}


# ============================================================
# LEVEL DEFINITIONS
# ============================================================

LEVELS = {
    "level1": {
        "title": "Level 1 - chuỗi A -> B -> C",
        "description": (
            "Map 5x5 de, mot chuoi A -> B -> C. Cham vao B hoac C truoc A se bi "
            "phat."
        ),
        "grid": [
            "#S...",
            "#.A..",
            "#..B.",
            "C....",
            "...X.",
        ],
        "chain": [["A"], ["B"], ["C"]],
        "aux": [],
        "locked": [],
    },
    "level2": {
        "title": "Level 2 - hai chuoi hop le khac nhau",
        "description": (
            "Map 7x7, bon vat pham A, B, C, D chap nhan hai chuoi hop le: "
            "A -> B -> D va A -> C -> D. Agent phai tu chon chuoi nao de di."
        ),
        "grid": [
            "#S....#",
            "#.A...#",
            "#..B..#",
            "#.....#",
            "#..C..#",
            "#.....#",
            "#...D.X",
        ],
        "chain": [["A"], ["B", "C"], ["D"]],
        "aux": [],
        "locked": [],
    },
    "level3": {
        "title": "Level 3 - vat pham bi khoa va o thuong",
        "description": (
            "Map 8x8, chuoi A -> B -> C -> D. Vat pham C bi khoa cho toi khi bat "
            "cong tac W, va o thuong K cho phep bo qua mot phan tu cua chuoi."
        ),
        "grid": [
            "########",
            "#S.....#",
            "#.A.K..#",
            "#..B...#",
            "#....W.#",
            "#....C.#",
            "#......#",
            "#D...X.#",
        ],
        "chain": [["A"], ["B"], ["C"], ["D"]],
        "aux": ["switch_on", "skip_used"],
        "locked": ["C"],
    },
}


def get_level(name):
    """Return the level configuration dictionary for the given level name."""
    if name not in LEVELS:
        raise KeyError(
            "Unknown level %r. Available levels: %s"
            % (name, ", ".join(sorted(LEVELS)))
        )
    return LEVELS[name]


# ============================================================
# ENVIRONMENT
# ============================================================

class OrderedTreasureHuntEnv(gym.Env):
    

    metadata = {"render_modes": ["ansi"]}

    def __init__(self, level="level1", rewards=None, max_steps=100):
        """Initialize the Ordered Treasure Hunt environment with map, rewards and rules."""
        spec = get_level(level)
        self.level = level
        self.title = spec["title"]
        self.description = spec["description"]
        self.grid = [list(row) for row in spec["grid"]]
        self.height = len(self.grid)
        self.width = len(self.grid[0])
        if any(len(row) != self.width for row in self.grid):
            raise ValueError(
                "Level %r has rows of different lengths: %s"
                % (level, [len(row) for row in self.grid])
            )
        self.chain = [list(stage) for stage in spec["chain"]]
        self.aux_names = list(spec["aux"])
        #: Items that cannot be taken before the switch is turned on.
        self.locked_items = list(spec.get("locked", []))
        self.n_aux = len(self.aux_names)
        self.n_stages = len(self.chain)
        self.max_steps = int(max_steps)
        self.rewards = dict(DEFAULT_REWARDS)
        if rewards:
            self.rewards.update(rewards)

        # --- item table: the bit order follows the chain order ---
        self.item_list = []
        for stage in self.chain:
            for letter in stage:
                if letter not in self.item_list:
                    self.item_list.append(letter)
        self.n_items = len(self.item_list)
        self._item_bit = {letter: 1 << i for i, letter in enumerate(self.item_list)}

        # --- tile table ---
        self._cell_items = {}
        self.start = None
        self.exit = None
        self.switch = None
        self.skip = None
        for r in range(self.height):
            for c in range(self.width):
                cell = self.grid[r][c]
                if cell == TILE_START:
                    self.start = (r, c)
                elif cell == TILE_EXIT:
                    self.exit = (r, c)
                elif cell == TILE_SWITCH:
                    self.switch = (r, c)
                elif cell == TILE_SKIP:
                    self.skip = (r, c)
                elif cell.isalpha() and cell.isupper():
                    self._cell_items[(r, c)] = cell
        if self.start is None or self.exit is None:
            raise ValueError("Level %r needs a start 'S' and an exit 'X'" % level)

        # --- state space: only the reachable states are enumerated ---
        self._states, self._index = self._build_state_space()
        self.n_states = len(self._states)
        self.start_index = self._index[
            (self.start[0], self.start[1], 0, 0)
        ]

        # --- gymnasium spaces ---
        self.action_space = spaces.Discrete(N_ACTIONS)
        self.observation_space = spaces.Discrete(self.n_states)

        self._state = self.start_index
        self._steps = 0

    # ------------------------------------------------------------
    # STATE ENCODING
    # ------------------------------------------------------------

    def _build_state_space(self):
        """Enumerate the reachable states with a breadth first search.

        Returns
        -------
        states : list
            The reachable state tuples, in discovery order.
        index : dict
            Maps a state tuple to its position inside ``states``.
        """
        start = (self.start[0], self.start[1], 0, 0)
        states = [start]
        index = {start: 0}
        queue = deque([start])
        while queue:
            state = queue.popleft()
            for action in range(N_ACTIONS):
                nxt, _reward, _term, _event = self._transition(state, action)
                if nxt not in index:
                    index[nxt] = len(states)
                    states.append(nxt)
                    queue.append(nxt)
        return states, index

    def state_to_index(self, state):
        """Convert a state tuple into its flat index."""
        row, col, progress, flags = state
        key = (row, col, progress, flags)
        if key not in self._index:
            raise KeyError("State %r is not reachable" % (key,))
        return self._index[key]

    def index_to_state(self, index):
        """Convert a flat index back into its state tuple."""
        return self._states[index]

    def decode(self, index):
        """Return a readable description of a flat state index."""
        row, col, progress, flags = self._states[index]
        items = [
            letter
            for i, letter in enumerate(self.item_list)
            if flags & (1 << i)
        ]
        aux = [
            name
            for j, name in enumerate(self.aux_names)
            if flags & (1 << (self.n_items + j))
        ]
        return {
            "row": row,
            "col": col,
            "task_progress": progress,
            "resolved_items": items,
            "aux": aux,
        }

    # ------------------------------------------------------------
    # DYNAMICS
    # ------------------------------------------------------------

    def _item_at(self, row, col, flags):
        """Return the item letter still present on a cell, or None.

        A resolved item disappears from the grid, therefore the tile becomes
        an ordinary floor tile.
        """
        letter = self._cell_items.get((row, col))
        if letter is None:
            return None
        if flags & self._item_bit[letter]:
            return None
        return letter

    def _aux_flag(self, flags, name):
        """Return True when the auxiliary variable ``name`` is set."""
        j = self.aux_names.index(name)
        return bool(flags & (1 << (self.n_items + j)))

    def _set_aux(self, flags, name):
        """Return the flags with the auxiliary variable ``name`` switched on."""
        j = self.aux_names.index(name)
        return flags | (1 << (self.n_items + j))

    def _transition(self, state, action):
        """Compute one deterministic transition of the environment.

        Parameters
        ----------
        state : tuple
            The current state ``(row, col, task_progress, flags)``.
        action : int
            The action to apply.

        Returns
        -------
        next_state : tuple
            The state reached by the action.
        reward : float
            The reward of the transition.
        terminated : bool
            True when the episode ends on the exit tile.
        event : str
            A short tag describing what happened. It is used by the report
            generators and by the auto play demo.
        """
        row, col, progress, flags = state
        rewards = self.rewards
        if progress >= self.n_stages and (row, col) == self.exit:
            # the episode is already finished, the state is absorbing
            return state, 0.0, True, "terminal"
        dr, dc = ACTION_DELTAS[action]
        nr, nc = row + dr, col + dc

        # --- move outside of the grid or into a wall ---
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return state, rewards["wall_penalty"], False, "wall"
        if self.grid[nr][nc] == TILE_WALL:
            return state, rewards["wall_penalty"], False, "wall"

        cell = self.grid[nr][nc]

        # --- an item tile ---
        letter = self._item_at(nr, nc, flags)
        if letter is not None:
            if progress >= self.n_stages:
                # every stage is resolved, no item can be left on the grid
                return state, rewards["wrong_item_penalty"], False, "wrong_item"
            if letter in self.chain[progress]:
                # correct next item of the chain
                new_flags = flags | self._item_bit[letter]
                new_state = (nr, nc, progress + 1, new_flags)
                return new_state, rewards["item_reward"], False, "collect"
            if letter in self.locked_items and not self._aux_flag(
                flags, "switch_on"
            ):
                # the item is locked until the switch is turned on
                return state, rewards["locked_penalty"], False, "locked"
            # out of order item, the move is blocked
            return state, rewards["wrong_item_penalty"], False, "wrong_item"

        # --- the exit tile ---
        if cell == TILE_EXIT:
            if progress >= self.n_stages:
                return (nr, nc, progress, flags), rewards["exit_bonus"], True, "exit"
            return state, rewards["early_exit_penalty"], False, "early_exit"

        # --- the switch tile ---
        if cell == TILE_SWITCH:
            new_flags = self._set_aux(flags, "switch_on")
            return (nr, nc, progress, new_flags), rewards["switch_reward"], False, "switch"

        # --- the bonus skip tile ---
        if cell == TILE_SKIP:
            if not self._aux_flag(flags, "skip_used") and progress < self.n_stages:
                # resolve the next chain element without collecting it
                letter = self.chain[progress][0]
                new_flags = self._set_aux(flags, "skip_used")
                new_flags = new_flags | self._item_bit[letter]
                new_state = (nr, nc, progress + 1, new_flags)
                return new_state, rewards["skip_reward"], False, "skip"
            # the tile has already been used, it is now an ordinary tile
            return (nr, nc, progress, flags), 0.0, False, "skip_used"

        # --- ordinary floor ---
        return (nr, nc, progress, flags), rewards["step_penalty"], False, "step"

    def build_transition_model(self):
        """Return the exact transition model of the environment.

        The result is a nested list ``P[state][action]`` where every entry is
        a list of tuples ``(probability, next_state, reward, terminated)``.
        The environment is deterministic, so every list holds exactly one
        tuple with probability 1.0.
        """
        model = []
        for state in self._states:
            row = []
            for action in range(N_ACTIONS):
                nxt, reward, terminated, _event = self._transition(state, action)
                row.append([(1.0, self.state_to_index(nxt), reward, terminated)])
            model.append(row)
        return model

    # ------------------------------------------------------------
    # GYMNASIUM API
    # ------------------------------------------------------------

    def reset(self, seed=None, options=None):
        """Reset the environment and return the first observation."""
        super().reset(seed=seed)
        self._state = self.start_index
        self._steps = 0
        return self._state, {"event": "reset"}

    def step(self, action):
        """Apply one action and return ``(obs, reward, terminated, truncated, info)``."""
        action = int(action)
        if not self.action_space.contains(action):
            raise ValueError("Action %r is not part of the action space" % (action,))
        state = self._states[self._state]
        nxt, reward, terminated, event = self._transition(state, action)
        self._state = self.state_to_index(nxt)
        self._steps += 1
        truncated = (not terminated) and (self._steps >= self.max_steps)
        info = {"event": event, "steps": self._steps, "state": self._state}
        return self._state, reward, terminated, truncated, info

    # ------------------------------------------------------------
    # TEXT RENDERING
    # ------------------------------------------------------------

    def grid_text(self, state=None):
        """Return the current grid as a list of text lines."""
        if state is None:
            state = self._states[self._state]
        row, col, progress, flags = state
        lines = []
        for r in range(self.height):
            chars = []
            for c in range(self.width):
                cell = self.grid[r][c]
                if r == row and c == col:
                    chars.append("@")
                    continue
                letter = self._item_at(r, c, flags)
                if letter is not None:
                    chars.append(letter)
                elif cell == TILE_WALL:
                    chars.append(TILE_WALL)
                else:
                    chars.append(cell)
            lines.append("".join(chars))
        return lines

    def policy_text(self, policy):
        """Return the greedy policy of every state as a list of text lines.

        One block of ``height`` lines is printed per value of
        ``task_progress``; the arrow of a cell is the action of the agent in
        the state ``(row, col, task_progress)``. When several reachable states
        share the same position and progress, the action of the first one is
        used.
        """
        # group the reachable states by (row, col, task_progress)
        groups = {}
        for index, (row, col, progress, _flags) in enumerate(self._states):
            groups.setdefault((row, col, progress), index)
        lines = []
        for progress in range(self.n_stages + 1):
            for r in range(self.height):
                chars = []
                for c in range(self.width):
                    cell = self.grid[r][c]
                    if cell == TILE_WALL:
                        chars.append(TILE_WALL)
                        continue
                    index = groups.get((r, c, progress))
                    if index is not None:
                        action = int(policy[index])
                        chars.append(ACTION_SYMBOLS[action])
                    elif cell == TILE_EXIT:
                        chars.append(TILE_EXIT)
                    elif cell == TILE_START:
                        chars.append(TILE_START)
                    else:
                        letter = self._cell_items.get((r, c))
                        chars.append(letter if letter else cell)
                lines.append("".join(chars))
            lines.append("")
        return lines


# ============================================================
# QUICK SELF CHECK
# ============================================================

### 2.1. Ba setting của đề tài

In [5]:
from envs import LEVELS, OrderedTreasureHuntEnv, ACTION_NAMES, ACTION_SYMBOLS

print("Danh sách hành động:", ACTION_NAMES)
print("Ký tự mô tả hướng  :", ACTION_SYMBOLS)
print()
for name in sorted(LEVELS):
    env = OrderedTreasureHuntEnv(level=name)
    print("%-8s |S| = %3d  |A| = %d  lưới %dx%d  chuỗi = %s  biến phụ = %s"
          % (name, env.n_states, env.action_space.n, env.height, env.width,
             env.chain, env.aux_names or "không có"))

Danh sách hành động: {0: 'up', 1: 'down', 2: 'left', 3: 'right'}
Ký tự mô tả hướng  : {0: '^', 1: 'v', 2: '<', 3: '>'}

level1   |S| =  79  |A| = 4  lưới 5x5  chuỗi = [['A'], ['B'], ['C']]  biến phụ = không có
level2   |S| = 199  |A| = 4  lưới 7x7  chuỗi = [['A'], ['B', 'C'], ['D']]  biến phụ = không có
level3   |S| = 693  |A| = 4  lưới 8x8  chuỗi = [['A'], ['B'], ['C'], ['D']]  biến phụ = ['switch_on', 'skip_used']


In [6]:
import matplotlib.pyplot as plt
import plots

levels = {name: OrderedTreasureHuntEnv(level=name) for name in sorted(LEVELS)}
plots.save_levels_figure(levels, "figures/notebook_levels.png", title="Ba cấu hình bản đồ của đề tài 08 (5x5, 7x7, 8x8)")
img = plt.imread("figures/notebook_levels.png")
plt.figure(figsize=(15, 5))
plt.imshow(img)
plt.axis("off")
plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_5272\797369938.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Mã nguồn các thuật toán (`agents.py`)

In [7]:

from __future__ import annotations

import time

import numpy as np


# ============================================================
# SMALL HELPERS
# ============================================================

def epsilon_greedy(q_values, state, epsilon, rng, n_actions):
    """Pick an action with the epsilon greedy rule.

    With probability ``1 - epsilon`` the action with the highest value is
    taken, otherwise a random action is drawn. Ties are broken randomly so
    that the exploration does not depend on the order of the actions.
    """
    best = np.flatnonzero(q_values[state] == q_values[state].max())
    if best.size > 1:
        greedy_action = int(rng.choice(best))
    else:
        greedy_action = int(best[0])
    if rng.random() < epsilon:
        return int(rng.integers(n_actions))
    return greedy_action


def greedy_policy(q_values):
    """Turn a table of action values into a deterministic greedy policy."""
    return np.argmax(q_values, axis=1).astype(int)


def epsilon_schedule(episode, n_episodes, epsilon_start, epsilon_end):
    """Linear decay of the exploration rate over the training episodes."""
    if n_episodes <= 1:
        return epsilon_end
    frac = episode / float(n_episodes - 1)
    return epsilon_start + (epsilon_end - epsilon_start) * frac


def evaluate_policy(env, policy, n_episodes=200, seed=1234):
    """Run the deterministic policy on the environment and collect statistics.

    Returns a dictionary with the mean and the standard deviation of the
    return, the success rate and the mean number of steps per episode.
    """
    returns = []
    lengths = []
    successes = 0
    for episode in range(n_episodes):
        state, _info = env.reset(seed=seed + episode)
        total = 0.0
        steps = 0
        terminated = False
        truncated = False
        while not (terminated or truncated):
            action = int(policy[state])
            state, reward, terminated, truncated, _info = env.step(action)
            total += reward
            steps += 1
        returns.append(total)
        lengths.append(steps)
        if terminated:
            successes += 1
    return {
        "eval_return_mean": float(np.mean(returns)),
        "eval_return_std": float(np.std(returns)),
        "success_rate": successes / float(n_episodes),
        "mean_steps": float(np.mean(lengths)),
        "min_steps": int(np.min(lengths)),
        "max_steps": int(np.max(lengths)),
    }


def rollout_events(env, policy, seed=0, max_steps=1000):
    """Play one episode and return the list of the events of the episode.

    The list tells which special tiles were used, for example ``skip`` when
    the bonus tile resolved a chain element.
    """
    state, _info = env.reset(seed=seed)
    events = []
    terminated = False
    truncated = False
    steps = 0
    while not (terminated or truncated) and steps < max_steps:
        action = int(policy[state])
        state, _reward, terminated, truncated, info = env.step(action)
        events.append(info.get("event", ""))
        steps += 1
    return events


# ============================================================
# BASE CLASS
# ============================================================

class BaseAgent:
    """Common interface of every algorithm of this project."""

    name = "base"
    #: True when the algorithm needs the transition model of the environment.
    needs_model = False

    def fit(self, env, seed=42, **kwargs):
        """Train the agent on ``env`` and return the result dictionary."""
        raise NotImplementedError


# ============================================================
# VALUE ITERATION
# ============================================================

class ValueIterationAgent(BaseAgent):
    """Value Iteration, a model based dynamic programming method.

    The Bellman optimality operator is applied until the maximum change of
    the value function drops below ``theta``:

        V(s) <- max_a sum_s' P(s'|s,a) [ r + gamma * V(s') ]
    """

    name = "vi"
    needs_model = True

    def fit(self, env, gamma=0.99, theta=1e-8, max_iterations=10000, **kwargs):
        """Run Value Iteration until Bellman convergence; returns policy and value dictionary."""
        started = time.perf_counter()
        model = env.build_transition_model()
        n_states = env.n_states
        n_actions = env.action_space.n
        value = np.zeros(n_states)
        deltas = []
        iterations = 0
        for iteration in range(max_iterations):
            new_value = np.zeros(n_states)
            for state in range(n_states):
                best = -np.inf
                for action in range(n_actions):
                    value_sum = 0.0
                    for prob, next_state, reward, terminated in model[state][action]:
                        future = 0.0 if terminated else gamma * value[next_state]
                        value_sum += prob * (reward + future)
                    if value_sum > best:
                        best = value_sum
                new_value[state] = best
            delta = float(np.max(np.abs(new_value - value)))
            deltas.append(delta)
            value = new_value
            iterations = iteration + 1
            if delta < theta:
                break
        q_values = self.q_from_value(model, value, gamma)
        policy = greedy_policy(q_values)
        return {
            "algo": self.name,
            "Q": q_values,
            "V": value,
            "policy": policy,
            "curve": deltas,
            "returns": [],
            "n_iterations": iterations,
            "time_sec": time.perf_counter() - started,
        }

    @staticmethod
    def q_from_value(model, value, gamma):
        """Compute the action values of every state from a value function."""
        n_states = len(model)
        n_actions = len(model[0])
        q_values = np.zeros((n_states, n_actions))
        for state in range(n_states):
            for action in range(n_actions):
                value_sum = 0.0
                for prob, next_state, reward, terminated in model[state][action]:
                    future = 0.0 if terminated else gamma * value[next_state]
                    value_sum += prob * (reward + future)
                q_values[state, action] = value_sum
        return q_values


# ============================================================
# POLICY ITERATION
# ============================================================

class PolicyIterationAgent(BaseAgent):
    """Policy Iteration, a model based dynamic programming method.

    Every outer iteration evaluates the current policy and then improves it.
    The loop stops as soon as the improved policy is identical to the old
    one.
    """

    name = "pi"
    needs_model = True

    def fit(self, env, gamma=0.99, theta=1e-8, max_iterations=1000, **kwargs):
        """Run Policy Iteration (evaluation and improvement); returns policy and value dictionary."""
        started = time.perf_counter()
        model = env.build_transition_model()
        n_states = env.n_states
        n_actions = env.action_space.n
        policy = np.zeros(n_states, dtype=int)
        value = np.zeros(n_states)
        deltas = []
        iterations = 0
        for iteration in range(max_iterations):
            value, evaluations = self.policy_evaluation(
                model, policy, gamma, theta
            )
            deltas.append(float(evaluations))
            q_values = ValueIterationAgent.q_from_value(model, value, gamma)
            new_policy = greedy_policy(q_values)
            iterations = iteration + 1
            if np.array_equal(policy, new_policy):
                break
            policy = new_policy
        q_values = ValueIterationAgent.q_from_value(model, value, gamma)
        policy = greedy_policy(q_values)
        return {
            "algo": self.name,
            "Q": q_values,
            "V": value,
            "policy": policy,
            "curve": deltas,
            "returns": [],
            "n_iterations": iterations,
            "time_sec": time.perf_counter() - started,
        }

    @staticmethod
    def policy_evaluation(model, policy, gamma, theta, max_iterations=10000):
        """Iterative evaluation of a fixed policy.

        Returns the value function and the number of sweeps that were
        needed to reach the requested precision.
        """
        n_states = len(model)
        value = np.zeros(n_states)
        for iteration in range(max_iterations):
            new_value = np.zeros(n_states)
            for state in range(n_states):
                value_sum = 0.0
                for prob, next_state, reward, terminated in model[state][policy[state]]:
                    future = 0.0 if terminated else gamma * value[next_state]
                    value_sum += prob * (reward + future)
                new_value[state] = value_sum
            delta = float(np.max(np.abs(new_value - value)))
            value = new_value
            if delta < theta:
                return value, iteration + 1
        return value, max_iterations


# ============================================================
# Q-LEARNING
# ============================================================

class QLearningAgent(BaseAgent):
    """Q-learning, an off policy temporal difference control method.

        Q(s,a) <- Q(s,a) + alpha [ r + gamma max_a' Q(s',a') - Q(s,a) ]

    The target uses the best action of the next state, therefore the agent
    behaves greedily with respect to the current estimate of the optimal
    action values even while it explores.
    """

    name = "qlearning"

    def fit(self, env, episodes=20000, gamma=0.99, alpha=0.1,
            epsilon_start=1.0, epsilon_end=0.05, seed=42, **kwargs):
        """Train Q-learning agent with epsilon-greedy exploration; returns policy and Q dictionary."""
        started = time.perf_counter()
        rng = np.random.default_rng(seed)
        n_states = env.n_states
        n_actions = env.action_space.n
        q_values = np.zeros((n_states, n_actions))
        returns = []
        for episode in range(episodes):
            epsilon = epsilon_schedule(
                episode, episodes, epsilon_start, epsilon_end
            )
            state, _info = env.reset(seed=seed + episode)
            total = 0.0
            terminated = False
            truncated = False
            while not (terminated or truncated):
                action = epsilon_greedy(q_values, state, epsilon, rng, n_actions)
                next_state, reward, terminated, truncated, _info = env.step(action)
                if terminated:
                    target = reward
                else:
                    target = reward + gamma * float(q_values[next_state].max())
                q_values[state, action] += alpha * (target - q_values[state, action])
                state = next_state
                total += reward
            returns.append(total)
        policy = greedy_policy(q_values)
        return {
            "algo": self.name,
            "Q": q_values,
            "V": q_values.max(axis=1),
            "policy": policy,
            "curve": returns,
            "returns": returns,
            "n_iterations": episodes,
            "time_sec": time.perf_counter() - started,
        }


# ============================================================
# SARSA
# ============================================================

class SarsaAgent(BaseAgent):
    """SARSA, an on policy temporal difference control method.

        Q(s,a) <- Q(s,a) + alpha [ r + gamma Q(s',a') - Q(s,a) ]

    The action of the next state is the action that is really taken by the
    behaviour policy, so the agent learns the value of the policy it is
    following instead of the value of the optimal policy.
    """

    name = "sarsa"

    def fit(self, env, episodes=20000, gamma=0.99, alpha=0.1,
            epsilon_start=1.0, epsilon_end=0.05, seed=42, **kwargs):
        """Train on-policy SARSA agent with epsilon-greedy decay; returns policy and Q dictionary."""
        started = time.perf_counter()
        rng = np.random.default_rng(seed)
        n_states = env.n_states
        n_actions = env.action_space.n
        q_values = np.zeros((n_states, n_actions))
        returns = []
        for episode in range(episodes):
            epsilon = epsilon_schedule(
                episode, episodes, epsilon_start, epsilon_end
            )
            state, _info = env.reset(seed=seed + episode)
            total = 0.0
            terminated = False
            truncated = False
            while not (terminated or truncated):
                action = epsilon_greedy(q_values, state, epsilon, rng, n_actions)
                next_state, reward, terminated, truncated, _info = env.step(action)
                if terminated:
                    target = reward
                else:
                    next_action = epsilon_greedy(
                        q_values, next_state, epsilon, rng, n_actions
                    )
                    target = reward + gamma * q_values[next_state, next_action]
                q_values[state, action] += alpha * (target - q_values[state, action])
                state = next_state
                total += reward
            returns.append(total)
        policy = greedy_policy(q_values)
        return {
            "algo": self.name,
            "Q": q_values,
            "V": q_values.max(axis=1),
            "policy": policy,
            "curve": returns,
            "returns": returns,
            "n_iterations": episodes,
            "time_sec": time.perf_counter() - started,
        }


# ============================================================
# MONTE CARLO CONTROL
# ============================================================

class MonteCarloAgent(BaseAgent):
    """On policy first visit Monte Carlo control.

    The complete episode is generated with the epsilon greedy behaviour
    policy, the returns are computed backwards and the first visit of every
    state is updated with the incremental average of the observed returns.
    There is no bootstrapping, so the algorithm needs a finished episode.
    """

    name = "mc"

    def fit(self, env, episodes=20000, gamma=0.99, alpha=0.1,
            epsilon_start=1.0, epsilon_end=0.05, seed=42, **kwargs):
        """Train on-policy first-visit Monte Carlo agent; returns policy and Q dictionary."""
        started = time.perf_counter()
        rng = np.random.default_rng(seed)
        n_states = env.n_states
        n_actions = env.action_space.n
        q_values = np.zeros((n_states, n_actions))
        counts = np.zeros((n_states, n_actions))
        returns = []
        for episode in range(episodes):
            epsilon = epsilon_schedule(
                episode, episodes, epsilon_start, epsilon_end
            )
            state, _info = env.reset(seed=seed + episode)
            trajectory = []
            total = 0.0
            terminated = False
            truncated = False
            while not (terminated or truncated):
                action = epsilon_greedy(q_values, state, epsilon, rng, n_actions)
                next_state, reward, terminated, truncated, _info = env.step(action)
                trajectory.append((state, action, reward))
                state = next_state
                total += reward
            returns.append(total)
            # compute the returns of the episode, backwards
            episode_returns = []
            accumulated = 0.0
            for _state, _action, reward in reversed(trajectory):
                accumulated = reward + gamma * accumulated
                episode_returns.append(accumulated)
            episode_returns.reverse()
            # first visit update with the incremental average
            visited = set()
            for (visited_state, visited_action, _reward), g in zip(
                trajectory, episode_returns
            ):
                if (visited_state, visited_action) in visited:
                    continue
                visited.add((visited_state, visited_action))
                counts[visited_state, visited_action] += 1.0
                q_values[visited_state, visited_action] += (
                    g - q_values[visited_state, visited_action]
                ) / counts[visited_state, visited_action]
        policy = greedy_policy(q_values)
        return {
            "algo": self.name,
            "Q": q_values,
            "V": q_values.max(axis=1),
            "policy": policy,
            "curve": returns,
            "returns": returns,
            "n_iterations": episodes,
            "time_sec": time.perf_counter() - started,
        }


# ============================================================
# REGISTRY
# ============================================================

#: All algorithms of the project, indexed by the name used on the command line.
AGENTS = {
    "vi": ValueIterationAgent,
    "pi": PolicyIterationAgent,
    "qlearning": QLearningAgent,
    "sarsa": SarsaAgent,
    "mc": MonteCarloAgent,
}

#: Pretty names used in the figures and in the reports.
ALGO_LABELS = {
    "vi": "Value Iteration",
    "pi": "Policy Iteration",
    "qlearning": "Q-learning",
    "sarsa": "SARSA",
    "mc": "Monte Carlo",
}

#: Short description of every algorithm, used by the report generators.
ALGO_NOTES = {
    "vi": "Model based, off policy, bootstraps with the Bellman optimality operator.",
    "pi": "Model based, off policy, alternates policy evaluation and policy improvement.",
    "qlearning": "Model free, off policy, bootstraps with max_a Q(s',a).",
    "sarsa": "Model free, on policy, bootstraps with Q(s',a') of the behaviour policy.",
    "mc": "Model free, on policy, no bootstrap, needs a complete episode.",
}


def build_agent(name):
    """Return a fresh agent instance for the given name."""
    if name not in AGENTS:
        raise KeyError(
            "Unknown algorithm %r. Available algorithms: %s"
            % (name, ", ".join(sorted(AGENTS)))
        )
    return AGENTS[name]()

## 4. Mã nguồn cấu hình (`configs.py`)

In [8]:
"""configs.py - Configuration of the experiments of De tai 08.

The module holds the dictionary ``CONFIGS`` with one entry per setting of the
assignment and the helper ``get_config(name)`` that returns a copy of a
configuration. Every configuration describes

    level            : key of the level inside ``envs.LEVELS``
    episodes         : number of training episodes of the model free methods
    eval_episodes    : number of episodes used to evaluate the learned policy
    seed             : seed of the training and of the evaluation
    reward_profile   : key of the reward profile inside ``REWARD_PROFILES``
    algorithms       : hyper parameters of the six algorithms

Keeping the configuration in one place makes the comparison reproducible and
lets the report generators read the same numbers that the training used.
"""

from __future__ import annotations

import copy


# ============================================================
# REWARD PROFILES
# ============================================================

#: Reward profiles used to show how the policy depends on the reward design.
#: The profiles only differ in the cost of one move, which decides whether
#: the detour of level 4 is worth it.
REWARD_PROFILES = {
    "default": {
        "step_penalty": -1.0,
        "item_reward": 10.0,
        "exit_bonus": 100.0,
        "wrong_item_penalty": -20.0,
    },
    "cheap_steps": {
        "step_penalty": -0.2,
        "item_reward": 10.0,
        "exit_bonus": 100.0,
        "wrong_item_penalty": -20.0,
    },
    "expensive_steps": {
        "step_penalty": -5.0,
        "item_reward": 10.0,
        "exit_bonus": 100.0,
        "wrong_item_penalty": -20.0,
    },
}


# ============================================================
# SHARED HYPER PARAMETERS
# ============================================================

#: Default hyper parameters of the six algorithms.
ALGO_PARAMS = {
    "vi": {"gamma": 0.99, "theta": 1e-8, "max_iterations": 10000},
    "pi": {"gamma": 0.99, "theta": 1e-8, "max_iterations": 1000},
    "qlearning": {
        "gamma": 0.99,
        "alpha": 0.1,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
    },
    "sarsa": {
        "gamma": 0.99,
        "alpha": 0.1,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
    },
    "mc": {
        "gamma": 0.99,
        "alpha": 0.1,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
    },
}

#: Names of the algorithms in the order they are reported.
ALGO_ORDER = ["vi", "pi", "qlearning", "sarsa", "mc"]


# ============================================================
# LEVEL CONFIGURATIONS
# ============================================================

CONFIGS = {
    "level1": {
        "level": "level1",
        "title": "Level 1 - chuỗi A -> B -> C",
        "description": (
            "Map 5x5 de, mot chuoi A -> B -> C. Cham vao mot vat pham sai thu tu "
            "bi phat, nen agent phai di vong quanh nhung vat pham con khoa."
        ),
        "episodes": 20000,
        "eval_episodes": 200,
        "seed": 42,
        "reward_profile": "default",
        "algorithms": {},
    },
    "level2": {
        "level": "level2",
        "title": "Level 2 - hai chuoi hop le khac nhau",
        "description": (
            "Map 7x7, bon vat pham A, B, C, D chap nhan hai chuoi hop le "
            "A -> B -> D va A -> C -> D. Cac vat pham da giai phai nam trong state, "
            "neu khong tien trinh se khong con tinh Markov."
        ),
        "episodes": 20000,
        "eval_episodes": 200,
        "seed": 42,
        "reward_profile": "default",
        "algorithms": {},
    },
    "level3": {
        "level": "level3",
        "title": "Level 3 - vat pham bi khoa va o thuong",
        "description": (
            "Map 8x8, chuoi A -> B -> C -> D. Vat pham C bi khoa cho toi khi bat "
            "cong tac W, va o thuong K bo qua duoc mot phan tu cua chuoi. Do do "
            "state can them bien phu switch_on va skip_used."
        ),
        "episodes": 20000,
        "eval_episodes": 200,
        "seed": 42,
        "reward_profile": "default",
        "algorithms": {},
    },
}


# ============================================================
# REWARD EXPERIMENT
# ============================================================

#: The reward experiment runs level 3 with two reward profiles. The only
#: difference is the cost of one move, and it is enough to flip the optimal
#: policy between "collect every item" and "use the bonus tile".
REWARD_EXPERIMENT = {
    "level": "level3",
    "profiles": ["cheap_steps", "expensive_steps"],
    "episodes": 20000,
    "eval_episodes": 200,
    "seed": 42,
    "algorithm": "qlearning",
}


# ============================================================
# SEED EXPERIMENT
# ============================================================

#: On the hard map the algorithms without a model can stop on the short path
#: and never try the long one again. This experiment repeats the training with
#: several random seeds, to see how often that happens.
SEED_EXPERIMENT = {
    "level": "level3",
    "seeds": [42, 7, 123],
    "episodes": 20000,
    "eval_episodes": 200,
    "algorithms": ["pi", "qlearning", "sarsa", "mc"],
}


# ============================================================
# HELPERS
# ============================================================

def get_config(name):
    """Return a copy of the configuration stored under ``name``."""
    if name not in CONFIGS:
        raise KeyError(
            "Unknown configuration %r. Available configurations: %s"
            % (name, ", ".join(sorted(CONFIGS)))
        )
    config = copy.deepcopy(CONFIGS[name])
    config["name"] = name
    config["rewards"] = copy.deepcopy(
        REWARD_PROFILES[config["reward_profile"]]
    )
    config["algorithms"] = {
        algo: copy.deepcopy(ALGO_PARAMS[algo]) for algo in ALGO_ORDER
    }
    return config


def get_reward_profile(name):
    """Return a copy of a reward profile."""
    if name not in REWARD_PROFILES:
        raise KeyError(
            "Unknown reward profile %r. Available profiles: %s"
            % (name, ", ".join(sorted(REWARD_PROFILES)))
        )
    return copy.deepcopy(REWARD_PROFILES[name])


def config_names():
    """Return the names of every configuration, ordered by difficulty."""
    return list(CONFIGS)

## 5. Mã nguồn vẽ đồ hoạ (`plots.py`)

In [9]:
"""plots.py - Figures of De tai 08.

The module draws the environment, the learned policies, the learning curves
and the comparison charts. Every figure is written as a PNG file into the
folder ``figures`` so that the notebook, the Word report and the PowerPoint
can reuse exactly the same pictures.

The text inside the figures is kept in English on purpose: the reports carry
the Vietnamese explanation, the figures stay readable for everybody.
"""

from __future__ import annotations

import os

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, FancyArrow, Rectangle

from envs import ACTION_DELTAS, TILE_EXIT, TILE_FLOOR, TILE_SKIP, TILE_START, TILE_SWITCH, TILE_WALL


# ============================================================
# COLORS
# ============================================================

COLOR_WALL = "#37474F"
COLOR_FLOOR = "#ECEFF1"
COLOR_GRID = "#CFD8DC"
COLOR_START = "#1E88E5"
COLOR_EXIT = "#43A047"
COLOR_SWITCH = "#FB8C00"
COLOR_SKIP = "#8E24AA"
COLOR_ITEM = "#FDD835"
COLOR_ITEM_DONE = "#CFD8DC"
COLOR_AGENT = "#E53935"
COLOR_ARROW = "#37474F"
COLOR_TEXT = "#263238"


# ============================================================
# LOW LEVEL DRAWING
# ============================================================

def _cell_center(row, col):
    """Return the centre of a cell in data coordinates."""
    return col + 0.5, row + 0.5


def draw_grid(ax, env, state=None, policy=None, agent=None, title=None,
              show_policy=True, progress=None):
    """Render the gridworld environment with tiles, agent, and policy arrows on axis ax."""

    Parameters
    ----------
    state : tuple, optional
        State tuple whose tiles are drawn. By default the current state of
        the environment is used.
    policy : array, optional
        Greedy policy; an arrow is drawn on every cell of the grid.
    progress : int, optional
        Value of ``task_progress`` whose arrows are drawn. By default the
        progress of the drawn state is used.
    agent : tuple, optional
        Position ``(row, col)`` of the agent, drawn on top of the grid.
    """
    if state is None:
        state = env._states[env._state]
    state_row, state_col, state_progress, flags = state
    if progress is None:
        progress = state_progress

    for r in range(env.height):
        for c in range(env.width):
            cell = env.grid[r][c]
            if cell == TILE_WALL:
                face = COLOR_WALL
            elif cell == TILE_EXIT:
                face = COLOR_EXIT
            elif cell == TILE_SWITCH:
                face = COLOR_SWITCH
            elif cell == TILE_SKIP:
                face = COLOR_SKIP
            elif cell == TILE_START:
                face = COLOR_START
            else:
                face = COLOR_FLOOR
            ax.add_patch(
                Rectangle(
                    (c, r), 1, 1, facecolor=face,
                    edgecolor=COLOR_GRID, linewidth=0.8
                )
            )

    # items, the resolved ones are drawn in grey
    for (r, c), letter in env._cell_items.items():
        resolved = bool(flags & env._item_bit[letter])
        ax.add_patch(
            Circle(
                _cell_center(r, c), 0.28,
                facecolor=COLOR_ITEM_DONE if resolved else COLOR_ITEM,
                edgecolor=COLOR_TEXT, linewidth=1.2
            )
        )
        ax.text(
            c + 0.5, r + 0.5, letter,
            ha="center", va="center", fontsize=11, fontweight="bold",
            color=COLOR_TEXT if not resolved else "#78909C"
        )

    # labels of the special tiles
    special = [
        (env.exit, "X", "white"),
        (env.switch, "W", "white"),
        (env.skip, "K", "white"),
        (env.start, "S", "white"),
    ]
    for position, text, color in special:
        if position is None:
            continue
        r, c = position
        ax.text(
            c + 0.5, r + 0.5, text,
            ha="center", va="center", fontsize=11, fontweight="bold", color=color
        )

    # policy arrows
    if policy is not None and show_policy:
        groups = {}
        for index, (sr, sc, sprogress, _flags) in enumerate(env._states):
            groups.setdefault((sr, sc, sprogress), index)
        for r in range(env.height):
            for c in range(env.width):
                if env.grid[r][c] == TILE_WALL:
                    continue
                index = groups.get((r, c, progress))
                if index is None:
                    continue
                action = int(policy[index])
                dr, dc = ACTION_DELTAS[action]
                x0, y0 = _cell_center(r, c)
                ax.add_patch(
                    FancyArrow(
                        x0, y0, 0.34 * dc, 0.34 * dr,
                        width=0.05, head_width=0.16, head_length=0.16,
                        color=COLOR_ARROW, alpha=0.75,
                        length_includes_head=True
                    )
                )

    # the agent
    if agent is None:
        agent = (state_row, state_col)
    ax.add_patch(
        Circle(
            _cell_center(agent[0], agent[1]), 0.22,
            facecolor=COLOR_AGENT, edgecolor="white", linewidth=1.2, zorder=5
        )
    )

    ax.set_xlim(0, env.width)
    ax.set_ylim(env.height, 0)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=11, color=COLOR_TEXT)
    return ax


def save_levels_figure(levels, path, title="Ba setting của đề tài 08"):
    """Save one picture that shows the three settings side by side.

    ``levels`` maps the name of a setting to the environment object.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    names = list(levels)
    figure, axes = plt.subplots(
        1, len(names), figsize=(4.2 * len(names), 4.6)
    )
    if len(names) == 1:
        axes = [axes]
    for ax, name in zip(axes, names):
        env = levels[name]
        draw_grid(ax, env, policy=None, title="%s  (|S| = %d)"
                  % (name, env.n_states))
    figure.suptitle(title, fontsize=12)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path


def save_environment_figure(env, path, policy=None, title=None, state=None):
    """Save a picture of the environment, optionally with the policy arrows.

    When a policy is given, one panel is drawn for every value of
    ``task_progress``, because the optimal action depends on it.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if policy is None:
        figure, ax = plt.subplots(figsize=(0.95 * env.width, 0.95 * env.height))
        draw_grid(ax, env, state=state, policy=policy, title=title)
        figure.tight_layout()
        figure.savefig(path, dpi=150)
        plt.close(figure)
        return path
    panels = env.n_stages + 1
    figure, axes = plt.subplots(
        1, panels, figsize=(0.95 * env.width * panels, 0.95 * env.height + 1.1)
    )
    if panels == 1:
        axes = [axes]
    for index, ax in enumerate(axes):
        draw_grid(ax, env, policy=policy, title="task_progress = %d" % index,
                  progress=index)
    figure.suptitle(title or env.title, fontsize=12)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path


# ============================================================
# LEARNING CURVES
# ============================================================

def moving_average(values, window):
    """Return the moving average of a sequence."""
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    window = max(1, min(window, values.size))
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")


def save_learning_curves(curves, path, title, window=200,
                         ylabel="return của một episode"):
    """Plot and save smoothed training learning curves for model-free algorithms."""
    """Save the learning curves of several algorithms on one figure.

    ``curves`` maps the name of an algorithm to its list of episode returns.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    figure, ax = plt.subplots(figsize=(8, 4.5))
    for name, values in curves.items():
        values = np.asarray(values, dtype=float)
        if values.size == 0:
            continue
        smooth = moving_average(values, window)
        offset = values.size - smooth.size
        ax.plot(
            np.arange(offset, values.size), smooth,
            label=name, linewidth=1.6
        )
    ax.set_xlabel("episode huấn luyện")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=9)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path


def save_dp_convergence(curves, path, title):
    """Save the convergence curves of the dynamic programming algorithms."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    figure, ax = plt.subplots(figsize=(8, 4.5))
    for name, values in curves.items():
        values = np.asarray(values, dtype=float)
        if values.size == 0:
            continue
        ax.plot(np.arange(1, values.size + 1), values, label=name, linewidth=1.6)
    ax.set_yscale("log")
    ax.set_xlabel("vòng lặp")
    ax.set_ylabel("độ lớn nhất |V(k+1) - V(k)|")
    ax.set_title(title, fontsize=12)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path


# ============================================================
# COMPARISON CHARTS
# ============================================================

def save_comparison_chart(runs, path, metric, title, ylabel):
    """Save a grouped bar chart that compares the algorithms on every level.

    ``runs`` is a list of dictionaries with the keys ``config``, ``algo`` and
    ``metric``.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    levels = []
    for run in runs:
        if run["config"] not in levels:
            levels.append(run["config"])
    algos = []
    for run in runs:
        if run["algo"] not in algos:
            algos.append(run["algo"])
    labels = {}
    for run in runs:
        labels[run["algo"]] = run.get("algo_label", run["algo"])
    width = 0.8 / max(1, len(algos))
    figure, ax = plt.subplots(figsize=(10, 4.8))
    positions = np.arange(len(levels))
    for index, algo in enumerate(algos):
        values = []
        for level in levels:
            found = None
            for run in runs:
                if run["config"] == level and run["algo"] == algo:
                    found = run[metric]
                    break
            values.append(0.0 if found is None else found)
        ax.bar(
            positions + index * width, values, width,
            label=labels[algo], alpha=0.85
        )
    ax.set_xticks(positions + width * (len(algos) - 1) / 2.0)
    ax.set_xticklabels(levels)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12)
    ax.grid(axis="y", alpha=0.3)
    highest = max([run[metric] for run in runs] + [1.0])
    ax.set_ylim(0, highest * 1.25)
    ax.legend(fontsize=9, ncol=len(algos), loc="upper center",
              bbox_to_anchor=(0.5, 1.0), framealpha=0.9)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path


def save_seed_comparison(experiment, path, title):
    """Save a bar chart of the return of every algorithm for every seed.

    ``experiment`` is the dictionary of the seed experiment: it holds the list
    of seeds and the list of runs with the keys ``algo``, ``seed`` and
    ``eval_return_mean``.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    seeds = experiment["seeds"]
    algos = []
    for run in experiment["runs"]:
        if run["algo"] not in algos:
            algos.append(run["algo"])
    labels = {}
    for run in experiment["runs"]:
        labels[run["algo"]] = run.get("algo_label", run["algo"])
    width = 0.8 / max(1, len(algos))
    figure, ax = plt.subplots(figsize=(10, 4.8))
    positions = np.arange(len(seeds))
    for index, algo in enumerate(algos):
        values = []
        for seed in seeds:
            found = None
            for run in experiment["runs"]:
                if run["algo"] == algo and run["seed"] == seed:
                    found = run["eval_return_mean"]
                    break
            values.append(0.0 if found is None else found)
        ax.bar(positions + index * width, values, width,
               label=labels[algo], alpha=0.85)
    ax.set_xticks(positions + width * (len(algos) - 1) / 2.0)
    ax.set_xticklabels(["seed %d" % seed for seed in seeds])
    ax.set_ylabel("mean return của policy tham lam")
    ax.set_title(title, fontsize=12)
    ax.grid(axis="y", alpha=0.3)
    highest = max([run["eval_return_mean"] for run in experiment["runs"]] + [1.0])
    ax.set_ylim(0, highest * 1.3)
    ax.legend(fontsize=9, ncol=len(algos), loc="upper center",
              bbox_to_anchor=(0.5, 1.0), framealpha=0.9)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path


def save_policy_comparison(env, entries, path, title):
    """Save a row of pictures that show one policy per entry.

    ``entries`` is a list of dictionaries with the keys ``name`` (the title of
    the panel), ``policy`` (the greedy policy), ``caption`` (the text that is
    printed under the picture) and ``progress`` (the value of
    ``task_progress`` whose arrows are drawn).
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    count = len(entries)
    figure, axes = plt.subplots(
        1, count, figsize=(0.95 * env.width * count, 0.95 * env.height + 1.4)
    )
    if count == 1:
        axes = [axes]
    for ax, entry in zip(axes, entries):
        draw_grid(ax, env, policy=entry["policy"], title=entry["name"],
                  progress=entry.get("progress"))
        ax.set_xlabel(entry.get("caption", ""), fontsize=8, color=COLOR_TEXT)
    figure.suptitle(title, fontsize=12)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path


# ============================================================
# MARKOV PROPERTY FIGURE
# ============================================================

def save_markov_figure(env, path, level="level1", cell=(2, 1), item_cell=(2, 2)):
    """Save the figure that shows why the task progress belongs to the state.

    The left panel shows the cell with a task progress of zero, where the
    item is still on the grid, the right panel shows the same cell with a
    task progress of one, where the item was collected and disappeared. The
    action towards the item therefore gives two different rewards, so the two
    situations cannot be the same state.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    states = {}
    for index, state in enumerate(env._states):
        row, col, progress, flags = state
        if (row, col) == cell and progress in (0, 1):
            states.setdefault(progress, (index, state))
    if 0 not in states or 1 not in states:
        raise ValueError(
            "The cell %r is not reachable with two different task_progress "
            "values" % (cell,)
        )
    model = env.build_transition_model()
    captions = {}
    for progress, (index, _state) in states.items():
        entries = []
        for action in range(env.action_space.n):
            for prob, next_state, reward, terminated in model[index][action]:
                next_row, next_col, next_progress, _flags = env._states[next_state]
                if (next_row, next_col) == item_cell:
                    entries.append((action, reward, next_progress))
        if entries:
            action, reward, next_progress = entries[0]
            captions[progress] = (
                "bước vào ô vật phẩm cho %+.0f và tiến độ thành %d"
                % (reward, next_progress)
            )
        else:
            captions[progress] = "không tới được vật phẩm từ ô này"
    figure, axes = plt.subplots(1, 2, figsize=(12, 5.8))
    for ax, progress in zip(axes, sorted(states)):
        index, state = states[progress]
        draw_grid(
            ax, env, state=state,
            title="trạng thái (%d, %d, %d)" % (cell[0], cell[1], progress)
        )
        ax.set_xlabel(captions[progress], fontsize=9, color=COLOR_TEXT)
    figure.suptitle(
        "Cùng một ô (%d, %d), task_progress khác nhau là hai state khác nhau"
        % (cell[0], cell[1]),
        fontsize=12,
    )
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path


# ============================================================
# AUTO PLAY ANIMATION
# ============================================================

def save_episode_animation(env, policy, path, title, fps=6, seed=42, max_steps=100):
    """Play one episode of the learned policy and store it as a GIF file.

    The agent is drawn on the grid, the resolved items disappear and the
    current reward of the episode is printed in the title of the figure.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    import matplotlib.animation as animation

    state, _info = env.reset(seed=seed)
    frames = []
    total = 0.0
    terminated = False
    truncated = False
    while not (terminated or truncated) and len(frames) < max_steps:
        action = int(policy[state])
        next_state, reward, terminated, truncated, info = env.step(action)
        total += reward
        state = next_state
        frames.append((state, total, info.get("event", ""), terminated))
    # keep the last frame a little longer so that the result is visible
    frames.extend([frames[-1]] * (fps * 2))

    figure, ax = plt.subplots(figsize=(1.15 * env.width, 1.15 * env.height))

    def update(frame_index):
        """Draw frame at frame_index for GIF animation; returns drawn artists."""
        ax.clear()
        current_state, running_total, event, done = frames[frame_index]
        draw_grid(ax, env, state=env._states[current_state], policy=policy,
                  show_policy=False)
        ax.set_title(
            "%s\nbước %d / %d   return = %.1f   sự kiện = %s%s"
            % (
                title, frame_index + 1, len(frames), running_total, event,
                "   (đã tới đích)" if done else "",
            ),
            fontsize=10,
        )
        return (ax,)

    anim = animation.FuncAnimation(figure, update, frames=len(frames), blit=False)
    writer = animation.PillowWriter(fps=fps)
    anim.save(path, writer=writer)
    plt.close(figure)
    return path


def save_trajectory_figure(env, policy, path, title, seed=42, max_steps=100):
    """Save the visited cells of one episode of the learned policy."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    state, _info = env.reset(seed=seed)
    visited = []
    total = 0.0
    terminated = False
    truncated = False
    while not (terminated or truncated) and len(visited) < max_steps:
        row, col, progress, flags = env._states[state]
        visited.append((row, col))
        action = int(policy[state])
        state, reward, terminated, truncated, _info = env.step(action)
        total += reward
    figure, ax = plt.subplots(figsize=(1.1 * env.width, 1.1 * env.height))
    draw_grid(ax, env, state=env._states[state], policy=policy, show_policy=False)
    for order, (r, c) in enumerate(visited):
        ax.text(
            c + 0.5, r + 0.22, str(order + 1),
            ha="center", va="center", fontsize=7, color="#B71C1C"
        )
    ax.set_title("%s\nreturn = %.1f, %d bước" % (title, total, len(visited)), fontsize=10)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)
    return path

## 6. Mã nguồn chạy lệnh (`main.py`)

In [10]:
"""main.py - Command line interface of De tai 08.

Examples
--------
Train one algorithm on one setting::

    python main.py --algo qlearning --config level1 --episodes 20000 --seed 42

Compare the six algorithms on the four settings and write the results::

    python main.py --compare --out results.json

Draw the environment with the learned policy::

    python main.py --render --config level3 --algo pi --out figures/policy_level3_pi.png

Play one episode with the learned policy and store it as a GIF::

    python main.py --autoplay --algo qlearning --config level1 --gif figures/autoplay_level1.gif
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time
from datetime import datetime

import numpy as np

# The Windows console uses cp1252, so printing Vietnamese would raise a
# UnicodeEncodeError. The output is switched to utf-8 here.
try:
    sys.stdout.reconfigure(encoding="utf-8")
except (AttributeError, ValueError):
    pass

from agents import (
    ALGO_LABELS,
    ALGO_NOTES,
    build_agent,
    evaluate_policy,
    rollout_events,
)
from configs import (
    ALGO_ORDER,
    ALGO_PARAMS,
    CONFIGS,
    REWARD_EXPERIMENT,
    REWARD_PROFILES,
    SEED_EXPERIMENT,
    config_names,
    get_config,
    get_reward_profile,
)
from envs import LEVELS, OrderedTreasureHuntEnv
import plots


# ============================================================
# CONSTANTS
# ============================================================

FIGURE_DIR = "figures"
RESULT_FILE = "results.json"
MAX_STEPS = 100
CURVE_POINTS = 200


# ============================================================
# HELPERS
# ============================================================

def make_env(config_name, reward_profile=None, max_steps=MAX_STEPS):
    """Build the environment of a configuration.

    Returns the environment, the configuration dictionary and the name of the
    reward profile that was used.
    """
    config = get_config(config_name)
    profile = reward_profile or config["reward_profile"]
    rewards = get_reward_profile(profile)
    env = OrderedTreasureHuntEnv(
        level=config["level"], rewards=rewards, max_steps=max_steps
    )
    return env, config, profile


def train_one(config_name, algo, episodes=None, seed=None,
              reward_profile=None, eval_episodes=None):
    """Train one algorithm on a config and evaluate policy; returns evaluation metrics dict."""
    """Train one algorithm and evaluate the policy it produced.

    Returns a dictionary with the metrics of the run. The keys that start
    with an underscore hold the objects that are needed by the figures and
    are removed before the results are written to the file.
    """
    env, config, profile = make_env(config_name, reward_profile)
    seed = config["seed"] if seed is None else seed
    episodes = config["episodes"] if episodes is None else episodes
    eval_episodes = config["eval_episodes"] if eval_episodes is None else eval_episodes
    params = dict(ALGO_PARAMS[algo])
    if algo in ("qlearning", "sarsa", "mc"):
        params["episodes"] = episodes
    agent = build_agent(algo)
    result = agent.fit(env, seed=seed, **params)
    metrics = evaluate_policy(
        env, result["policy"], n_episodes=eval_episodes, seed=seed + 100000
    )
    returns = np.asarray(result["returns"], dtype=float)
    events = rollout_events(env, result["policy"], seed=seed)
    summary = {
        "config": config_name,
        "level": config["level"],
        "algo": algo,
        "algo_label": ALGO_LABELS[algo],
        "algo_note": ALGO_NOTES[algo],
        "reward_profile": profile,
        "n_states": int(env.n_states),
        "n_actions": int(env.action_space.n),
        "n_stages": int(env.n_stages),
        "grid": "%dx%d" % (env.height, env.width),
        "episodes": int(episodes),
        "eval_episodes": int(eval_episodes),
        "seed": int(seed),
        "gamma": float(params.get("gamma", 0.99)),
        "train_return_mean": float(returns.mean()) if returns.size else None,
        "train_return_std": float(returns.std()) if returns.size else None,
        "n_iterations": int(result["n_iterations"]),
        "time_sec": float(result["time_sec"]),
        "uses_skip_tile": "skip" in events,
        "reaches_exit": "exit" in events,
    }
    summary.update(metrics)
    summary["curve"] = downsample(result["curve"], CURVE_POINTS)
    # objects that are only needed while the figures are built
    summary["_policy"] = result["policy"]
    summary["_curve_full"] = result["curve"]
    summary["_policy_text"] = env.policy_text(result["policy"])
    return summary


def downsample(values, points):
    """Reduce a long curve to at most ``points`` values."""
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return []
    if values.size <= points:
        return [float(v) for v in values]
    step = values.size / float(points)
    picked = [values[int(i * step)] for i in range(points)]
    return [float(v) for v in picked]


def strip_artifacts(summary):
    """Remove the private objects of a run so that it can be serialised."""
    clean = {key: value for key, value in summary.items()
             if not key.startswith("_")}
    return clean


# ============================================================
# MODES
# ============================================================

def run_comparison(episodes=None, seed=None, out=None, figures=True):
    """Train every algorithm on every setting and collect the results.

    The function is used by the command line and by the notebook, so both
    produce exactly the same numbers.

    Returns the dictionary that is written to the results file.
    """
    all_runs = []
    artifacts = {}
    levels_info = {}
    for config_name in config_names():
        env, config, _profile = make_env(config_name)
        levels_info[config_name] = {
            "title": config["title"],
            "description": config["description"],
            "level": config["level"],
            "height": int(env.height),
            "width": int(env.width),
            "n_states": int(env.n_states),
            "n_actions": int(env.action_space.n),
            "n_stages": int(env.n_stages),
            "chain": env.chain,
            "aux": env.aux_names,
        }
        print("=" * 74)
        print("Cấu hình %s : %s" % (config_name, config["title"]))
        print("  |S| = %d, |A| = %d, lưới %dx%d"
              % (env.n_states, env.action_space.n, env.height, env.width))
        artifacts[config_name] = {}
        for algo in ALGO_ORDER:
            started = time.perf_counter()
            summary = train_one(
                config_name, algo, episodes=episodes, seed=seed
            )
            artifacts[config_name][algo] = summary
            all_runs.append(strip_artifacts(summary))
            print(
                "  %-18s return %8.2f +/- %6.2f   success %5.1f %%   "
                "steps %5.1f   %6.2f s"
                % (
                    summary["algo_label"],
                    summary["eval_return_mean"],
                    summary["eval_return_std"],
                    100.0 * summary["success_rate"],
                    summary["mean_steps"],
                    time.perf_counter() - started,
                )
            )
    print("=" * 74)
    print("Thí nghiệm reward trên %s" % REWARD_EXPERIMENT["level"])
    reward_data = run_reward_experiment(episodes, seed, artifacts)
    print("=" * 74)
    print("Thí nghiệm seed trên %s" % SEED_EXPERIMENT["level"])
    seed_data = run_seed_experiment(episodes)
    results = {
        "meta": {
            "generated_at": datetime.now().isoformat(timespec="seconds"),
            "episodes": episodes,
            "seed": seed,
            "max_steps": MAX_STEPS,
            "algorithms": list(ALGO_ORDER),
            "algorithm_labels": dict(ALGO_LABELS),
            "algorithm_notes": dict(ALGO_NOTES),
            "reward_profiles": REWARD_PROFILES,
        },
        "levels": levels_info,
        "runs": all_runs,
        "reward_experiment": reward_data,
        "seed_experiment": seed_data,
    }
    out = out or RESULT_FILE
    with open(out, "w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2, ensure_ascii=False)
    print("=" * 74)
    print("Kết quả đã lưu vào %s" % out)
    if figures:
        build_figures(results, artifacts)
    return results


def run_reward_experiment(episodes, seed, artifacts=None):
    """Compare the two reward profiles on level 4.

    For every profile the optimal policy (Policy Iteration) and the policy
    that Q-learning learns are measured, so that the effect of the reward on
    the behaviour of the agent is visible.
    """
    data = {
        "level": REWARD_EXPERIMENT["level"],
        "algorithm": REWARD_EXPERIMENT["algorithm"],
        "profiles": [],
    }
    for profile in REWARD_EXPERIMENT["profiles"]:
        entry = {
            "profile": profile,
            "rewards": REWARD_PROFILES[profile],
        }
        for tag, algo in (("optimal", "pi"), ("learned", REWARD_EXPERIMENT["algorithm"])):
            summary = train_one(
                REWARD_EXPERIMENT["level"],
                algo,
                episodes=episodes,
                seed=seed,
                reward_profile=profile,
                eval_episodes=REWARD_EXPERIMENT["eval_episodes"],
            )
            if artifacts is not None:
                artifacts.setdefault("reward_" + profile, {})[algo] = summary
            entry[tag] = {
                "algo": algo,
                "eval_return_mean": summary["eval_return_mean"],
                "eval_return_std": summary["eval_return_std"],
                "success_rate": summary["success_rate"],
                "mean_steps": summary["mean_steps"],
                "uses_skip_tile": summary["uses_skip_tile"],
            }
        data["profiles"].append(entry)
        print("  hồ sơ reward %-16s tối ưu: return %8.2f bước %5.1f dùng ô K %-5s"
              "   học được: return %8.2f bước %5.1f dùng ô K %-5s"
              % (
                  profile,
                  entry["optimal"]["eval_return_mean"],
                  entry["optimal"]["mean_steps"],
                  entry["optimal"]["uses_skip_tile"],
                  entry["learned"]["eval_return_mean"],
                  entry["learned"]["mean_steps"],
                  entry["learned"]["uses_skip_tile"],
              ))
    return data


def run_seed_experiment(episodes):
    """Train the algorithms of the hard map with several random seeds.

    On the hard map the algorithms without a model can stop on the short path
    and never try the long one again. This experiment shows how often that
    happens, and if it depends on the random seed.
    """
    data = {
        "level": SEED_EXPERIMENT["level"],
        "seeds": list(SEED_EXPERIMENT["seeds"]),
        "runs": [],
    }
    for algo in SEED_EXPERIMENT["algorithms"]:
        for seed in SEED_EXPERIMENT["seeds"]:
            summary = train_one(
                SEED_EXPERIMENT["level"], algo, episodes=episodes, seed=seed,
                eval_episodes=SEED_EXPERIMENT["eval_episodes"],
            )
            data["runs"].append({
                "algo": algo,
                "algo_label": summary["algo_label"],
                "seed": seed,
                "eval_return_mean": summary["eval_return_mean"],
                "mean_steps": summary["mean_steps"],
                "uses_skip_tile": summary["uses_skip_tile"],
            })
            print("  seed %-4d %-18s return %8.2f  steps %5.1f  dùng ô thưởng: %s"
                  % (seed, summary["algo_label"],
                     summary["eval_return_mean"], summary["mean_steps"],
                     summary["uses_skip_tile"]))
    return data


def build_figures(results, artifacts):
    """Create every figure of the report from the results of the comparison."""
    os.makedirs(FIGURE_DIR, exist_ok=True)
    runs = results["runs"]
    levels = {}
    for config_name in config_names():
        env, _config, _profile = make_env(config_name)
        levels[config_name] = env
    plots.save_levels_figure(
        levels, os.path.join(FIGURE_DIR, "notebook_levels.png")
    )
    for config_name in config_names():
        env = levels[config_name]
        subset = artifacts[config_name]
        # the policy of Policy Iteration is the optimal one
        plots.save_environment_figure(
            env,
            os.path.join(FIGURE_DIR, "env_%s.png" % config_name),
            policy=subset["pi"]["_policy"],
            title="%s  (Policy Iteration)" % env.title,
        )
        # learning curves of the model free algorithms
        curves = {}
        for algo in ("qlearning", "sarsa", "mc"):
            curves[ALGO_LABELS[algo]] = subset[algo]["_curve_full"]
        plots.save_learning_curves(
            curves,
            os.path.join(FIGURE_DIR, "learning_curves_%s.png" % config_name),
            "Đường học - %s" % env.title,
        )
        # convergence of the dynamic programming methods
        dp_curves = {}
        for algo in ("vi", "pi"):
            dp_curves[ALGO_LABELS[algo]] = subset[algo]["_curve_full"]
        plots.save_dp_convergence(
            dp_curves,
            os.path.join(FIGURE_DIR, "dp_convergence_%s.png" % config_name),
            "Hội tụ của các phương pháp quy hoạch động - %s" % env.title,
        )
    # comparison charts
    plots.save_comparison_chart(
        runs,
        os.path.join(FIGURE_DIR, "comparison_return.png"),
        "eval_return_mean",
        "Mean return của mọi thuật toán trên mọi setting",
        "mean return của policy tham lam",
    )
    plots.save_comparison_chart(
        runs,
        os.path.join(FIGURE_DIR, "comparison_success_rate.png"),
        "success_rate",
        "Tỷ lệ thành công của mọi thuật toán trên mọi setting",
        "tỷ lệ thành công",
    )
    # markov property
    env1, _c, _p = make_env("level1")
    plots.save_markov_figure(
        env1, os.path.join(FIGURE_DIR, "markov_proof.png"), level="level1"
    )
    # reward profiles
    env3, _c, _p = make_env("level3")
    entries = []
    for entry in results["reward_experiment"]["profiles"]:
        for tag, algo in (("optimal", "pi"), ("learned", REWARD_EXPERIMENT["algorithm"])):
            summary = artifacts["reward_" + entry["profile"]][algo]
            entries.append({
                "name": "%s - %s" % (entry["profile"], tag),
                "policy": summary["_policy"],
                "progress": 1,
                "caption": "step penalty = %.1f\nreturn = %.1f, steps = %.1f\nbonus tile used: %s"
                           % (
                               entry["rewards"]["step_penalty"],
                               entry[tag]["eval_return_mean"],
                               entry[tag]["mean_steps"],
                               entry[tag]["uses_skip_tile"],
                           ),
            })
    plots.save_policy_comparison(
        env3,
        entries,
        os.path.join(FIGURE_DIR, "reward_profiles_level3.png"),
        "Level 3 - policy sau khi lấy vật phẩm A phụ thuộc chi phí của một bước",
    )
    # the failure case of Monte Carlo on level 3
    mc_summary = artifacts["level3"]["mc"]
    pi_summary = artifacts["level3"]["pi"]
    plots.save_policy_comparison(
        env3,
        [
            {
                "name": "Policy Iteration (tối ưu)",
                "policy": pi_summary["_policy"],
                "progress": 1,
                "caption": "return = %.1f, steps = %.1f\ndùng ô thưởng: %s"
                           % (
                               pi_summary["eval_return_mean"],
                               pi_summary["mean_steps"],
                               pi_summary["uses_skip_tile"],
                           ),
            },
            {
                "name": "Monte Carlo (học được)",
                "policy": mc_summary["_policy"],
                "progress": 1,
                "caption": "return = %.1f, steps = %.1f\ndùng ô thưởng: %s"
                           % (
                               mc_summary["eval_return_mean"],
                               mc_summary["mean_steps"],
                               mc_summary["uses_skip_tile"],
                           ),
            },
        ],
        os.path.join(FIGURE_DIR, "local_optimum_level3.png"),
        "Level 3 - một cực điểm cục bộ của Monte Carlo control",
    )
    # the result of the seed experiment
    plots.save_seed_comparison(
        results["seed_experiment"],
        os.path.join(FIGURE_DIR, "seed_experiment_level3.png"),
        "Level 3 - mean return của mỗi thuật toán theo từng seed",
    )
    # the trajectory of the auto play, used by the report and by the slides
    for config_name in ("level1", "level3"):
        env = levels[config_name]
        plots.save_trajectory_figure(
            env,
            artifacts[config_name]["qlearning"]["_policy"],
            os.path.join(
                FIGURE_DIR, "trajectory_%s_qlearning.png" % config_name
            ),
            env.title,
            seed=artifacts[config_name]["qlearning"]["seed"],
        )
    print("Các hình đã lưu vào %s" % FIGURE_DIR)




## 7. Chứng minh tính Markov bằng số liệu

Cùng một ô `(2, 1)` của level 1 được xét với hai giá trị khác nhau của
`task_progress`. Bằng mô hình chuyển trạng thái, ta tính giá trị của mỗi hành động
tại ô đó.

In [11]:
env = OrderedTreasureHuntEnv(level="level1")
model = env.build_transition_model()

for progress in (0, 1):
    index = None
    for i, (r, c, p, f) in enumerate(env._states):
        if (r, c) == (2, 1) and p == progress:
            index = i
            break
    print("trạng thái (2, 1, %d) - flags = %d" % (progress, env._states[index][3]))
    for action in range(env.action_space.n):
        for prob, nxt, reward, terminated in model[index][action]:
            nr, nc, np_, nf = env._states[nxt]
            print("   hành động %-5s -> (%d, %d, %d)  thưởng %+6.1f  %s"
                  % (ACTION_NAMES[action], nr, nc, np_, reward,
                     "kết thúc" if terminated else ""))
    print()

trạng thái (2, 1, 0) - flags = 0
   hành động up    -> (1, 1, 0)  thưởng   -1.0  
   hành động down  -> (3, 1, 0)  thưởng   -1.0  
   hành động left  -> (2, 1, 0)  thưởng   -3.0  
   hành động right -> (2, 2, 0)  thưởng   -1.0  

trạng thái (2, 1, 1) - flags = 1
   hành động up    -> (1, 1, 1)  thưởng   -1.0  
   hành động down  -> (3, 1, 1)  thưởng   -1.0  
   hành động left  -> (2, 1, 1)  thưởng   -3.0  
   hành động right -> (2, 2, 1)  thưởng   -1.0  



In [12]:
import plots

plots.save_markov_figure(env, "figures/markov_proof.png", level="level1")
plt.imshow(plt.imread("figures/markov_proof.png"))
plt.axis("off")
plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_5272\3618847457.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. So sánh 5 thuật toán trên 3 setting

Sau đây chạy lại toàn bộ thí nghiệm: 5 thuật toán x 3 setting, mỗi thuật toán dùng
cùng một seed để kết quả có thể lặp lại. Kết quả được ghi vào `results.json`, cũng
là tệp được đọc bởi báo cáo Word và file PowerPoint trình bày.

In [13]:
from main import run_comparison, train_one
from configs import config_names, ALGO_ORDER

EPISODES = 20000
SEED = 42

results = run_comparison(episodes=EPISODES, seed=SEED, out="results.json", figures=True)
print("Kết quả đã được lưu vào results.json")

Cấu hình level1 : Level 1 - chuỗi A -> B -> C
  |S| = 79, |A| = 4, lưới 5x5
  Value Iteration    return   122.00 +/-   0.00   success 100.0 %   steps  12.0     0.01 s
  Policy Iteration   return   122.00 +/-   0.00   success 100.0 %   steps  12.0     0.15 s
  Q-learning         return   122.00 +/-   0.00   success 100.0 %   steps  12.0     8.99 s
  SARSA              return   122.00 +/-   0.00   success 100.0 %   steps  12.0     9.76 s
  Monte Carlo        return   122.00 +/-   0.00   success 100.0 %   steps  12.0     6.42 s
Cấu hình level2 : Level 2 - hai chuoi hop le khac nhau
  |S| = 199, |A| = 4, lưới 7x7
  Value Iteration    return   123.00 +/-   0.00   success 100.0 %   steps  11.0     0.01 s
  Policy Iteration   return   123.00 +/-   0.00   success 100.0 %   steps  11.0     0.65 s
  Q-learning         return   123.00 +/-   0.00   success 100.0 %   steps  11.0     8.55 s
  SARSA              return   123.00 +/-   0.00   success 100.0 %   steps  11.0     9.70 s
  Monte Carlo      

In [14]:
table = pd.DataFrame(results["runs"])
table = table.pivot_table(index="algo", columns="config", values="eval_return_mean")
table = table.reindex(ALGO_ORDER)
table.index = [results["meta"]["algorithm_labels"][a] for a in table.index]
print("Mean return của policy tham lam trên mỗi setting")
display(table.round(2))

Mean return của policy tham lam trên mỗi setting


config,level1,level2,level3
Value Iteration,122.0,123.0,128.0
Policy Iteration,122.0,123.0,128.0
Q-learning,122.0,123.0,120.0
SARSA,122.0,123.0,120.0
Monte Carlo,122.0,123.0,119.0


In [15]:
success = pd.DataFrame(results["runs"]).pivot_table(
    index="algo", columns="config", values="success_rate").reindex(ALGO_ORDER)
steps = pd.DataFrame(results["runs"]).pivot_table(
    index="algo", columns="config", values="mean_steps").reindex(ALGO_ORDER)
print("Tỷ lệ thành công (%)")
display((success * 100).round(1))
print()
print("Số bước trung bình đến khi thoát")
display(steps.round(1))

Tỷ lệ thành công (%)


config,level1,level2,level3
algo,,,
vi,100.0,100.0,100.0
pi,100.0,100.0,100.0
qlearning,100.0,100.0,100.0
sarsa,100.0,100.0,100.0
mc,100.0,100.0,100.0



Số bước trung bình đến khi thoát


config,level1,level2,level3
algo,,,
vi,12.0,11.0,18.0
pi,12.0,11.0,18.0
qlearning,12.0,11.0,18.0
sarsa,12.0,11.0,18.0
mc,12.0,11.0,18.0


In [16]:
sizes = pd.DataFrame([
    {
        "setting": name,
        "chuỗi": str(info["chain"]),
        "biến phụ": ", ".join(info["aux"]) if info["aux"] else "không có",
        "lưới": "%dx%d" % (info["height"], info["width"]),
        "|S|": info["n_states"],
        "|A|": info["n_actions"],
    }
    for name, info in results["levels"].items()
])
display(sizes)

,setting,chuỗi,biến phụ,lưới,|S|,|A|
0,level1,"[['A'], ['B'], ['C']]",không có,5x5,79,4
1,level2,"[['A'], ['B', 'C'], ['D']]",không có,7x7,199,4
2,level3,"[['A'], ['B'], ['C'], ['D']]","switch_on, skip_used",8x8,693,4


### 8.1. Đường học của các thuật toán không mô hình

In [17]:
for name in config_names():
    plt.figure(figsize=(8, 4.2))
    plt.imshow(plt.imread("figures/learning_curves_%s.png" % name))
    plt.axis("off")
    plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_5272\3849884477.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.2. Hội tụ của Value Iteration và Policy Iteration

In [18]:
for name in config_names():
    plt.figure(figsize=(8, 4.2))
    plt.imshow(plt.imread("figures/dp_convergence_%s.png" % name))
    plt.axis("off")
    plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_5272\872217053.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.3. Policy tối ưu của 3 setting

In [19]:
for name in config_names():
    plt.figure(figsize=(11, 4.6))
    plt.imshow(plt.imread("figures/env_%s.png" % name))
    plt.axis("off")
    plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_5272\2170961337.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Phân tích trường hợp thất bại

Trên level 3, chuỗi `A -> B -> C -> D` dài hơn, và ô thưởng `K` nằm ngay cạnh vật
phẩm `A` nên agent rất dễ bước vào. Dùng ô thưởng thì không lấy được vật phẩm `B`, nên
agent phải đi vòng mới lấy được đủ bốn vật phẩm. Monte Carlo dùng first visit và cập
nhật bằng trung bình, nên nó dễ rơi vào một cực điểm cục bộ: giữ nguyên đường đi qua
ô thưởng và không thử lại đo vòng. Value Iteration và Policy Iteration biết trước mô
hình nên luôn tìm ra đường tối ưu. Q-learning thoát ra khỏi cực điểm này ở hai seed
trong ba seed, còn SARSA và Monte Carlo thì không.

In [20]:
plt.figure(figsize=(11, 4.8))
plt.imshow(plt.imread("figures/local_optimum_level3.png"))
plt.axis("off")
plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_5272\1054783679.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Thí nghiệm nhiều seed trên level 3

Vì kết quả phụ thuộc vào seed, chúng tôi chạy lại level 3 với ba seed khác nhau.
Bảng dưới cho thấy thuật toán nào thoát ra khỏi cực điểm cục bộ.

In [21]:
rows = []
for run in results["seed_experiment"]["runs"]:
    rows.append({
        "thuật toán": run["algo_label"],
        "seed": run["seed"],
        "mean return": run["eval_return_mean"],
        "số bước": run["mean_steps"],
        "dùng ô thưởng K": run["uses_skip_tile"],
    })
display(pd.DataFrame(rows))

,thuật toán,seed,mean return,số bước,dùng ô thưởng K
0,Policy Iteration,42,128.0,18.0,False
1,Policy Iteration,7,128.0,18.0,False
2,Policy Iteration,123,128.0,18.0,False
3,Q-learning,42,120.0,18.0,True
4,Q-learning,7,128.0,18.0,False
5,Q-learning,123,128.0,18.0,False
6,SARSA,42,120.0,18.0,True
7,SARSA,7,120.0,18.0,True
8,SARSA,123,120.0,18.0,True
9,Monte Carlo,42,119.0,18.0,True


In [22]:
plt.figure(figsize=(10, 4.8))
plt.imshow(plt.imread("figures/seed_experiment_level3.png"))
plt.axis("off")
plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_5272\3310876056.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Hai cấu hình reward khác nhau

Chỉ thay đổi chi phí của một bước, policy tối ưu của level 3 đổi ngay:

- `cheap_steps` (`step_penalty = -0.2`): bỏ qua một vật phẩm không đáng, agent đi vòng
  để lấy được cả bốn vật phẩm.
- `expensive_steps` (`step_penalty = -5.0`): bỏ qua một vật phẩm là tốt hơn, agent dùng
  ô thưởng `K` và rút ngắn đường đi.

In [23]:
rows = []
for entry in results["reward_experiment"]["profiles"]:
    for tag in ("optimal", "learned"):
        info = entry[tag]
        rows.append({
            "cấu hình reward": entry["profile"],
            "chi phí 1 bước": entry["rewards"]["step_penalty"],
            "loại policy": "tối ưu (PI)" if tag == "optimal" else "học được (Q-learning)",
            "mean return": info["eval_return_mean"],
            "số bước": info["mean_steps"],
            "dùng ô thưởng K": info["uses_skip_tile"],
        })
display(pd.DataFrame(rows))

,cấu hình reward,chi phí 1 bước,loại policy,mean return,số bước,dùng ô thưởng K
0,cheap_steps,-0.2,tối ưu (PI),137.6,18.0,False
1,cheap_steps,-0.2,học được (Q-learning),137.6,18.0,False
2,expensive_steps,-5.0,tối ưu (PI),80.0,18.0,True
3,expensive_steps,-5.0,học được (Q-learning),80.0,18.0,True


In [24]:
plt.figure(figsize=(13, 4.8))
plt.imshow(plt.imread("figures/reward_profiles_level3.png"))
plt.axis("off")
plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_5272\131079714.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Chế độ tự chơi (auto play)

Agent chạy một episode bằng policy đã học, đồng thời ghi lại hình ảnh động họa.
Video được lưu thành file GIF trong thư mục `figures`.

In [25]:
import plots
from agents import build_agent, rollout_events
from configs import ALGO_PARAMS

env = OrderedTreasureHuntEnv(level="level1")
result = build_agent("qlearning").fit(env, episodes=20000, seed=42, **ALGO_PARAMS["qlearning"])
policy = result["policy"]

state, _ = env.reset(seed=42)
total = 0.0
print("bước | ô (row,col) | tiến độ | hành động | thưởng | sự kiện")
for step in range(1, 40):
    action = int(policy[state])
    row, col, progress, flags = env._states[state]
    state, reward, terminated, truncated, info = env.step(action)
    total += reward
    print("%4d | (%d, %d)       | %8d | %-8s | %+6.1f | %s"
          % (step, row, col, progress, ACTION_NAMES[action], reward, info["event"]))
    if terminated or truncated:
        break
print("Tổng return = %.1f" % total)

bước | ô (row,col) | tiến độ | hành động | thưởng | sự kiện
   1 | (0, 1)       |        0 | down     |   -1.0 | step
   2 | (1, 1)       |        0 | right    |  +10.0 | collect
   3 | (1, 2)       |        1 | down     |   -1.0 | step
   4 | (2, 2)       |        1 | right    |  +10.0 | collect
   5 | (2, 3)       |        2 | down     |   -1.0 | step
   6 | (3, 3)       |        2 | left     |   -1.0 | step
   7 | (3, 2)       |        2 | left     |   -1.0 | step
   8 | (3, 1)       |        2 | left     |  +10.0 | collect
   9 | (3, 0)       |        3 | down     |   -1.0 | step
  10 | (4, 0)       |        3 | right    |   -1.0 | step
  11 | (4, 1)       |        3 | right    |   -1.0 | step
  12 | (4, 2)       |        3 | right    | +100.0 | exit
Tổng return = 122.0


In [26]:
plots.save_episode_animation(
    env, policy, "figures/autoplay_level1.gif",
    "Level 1 - Q-learning", fps=6, seed=42,
)
print("Đã lưu figures/autoplay_level1.gif")

Đã lưu figures/autoplay_level1.gif


In [27]:
plt.figure(figsize=(7, 7.5))
plt.imshow(plt.imread("figures/autoplay_level1.gif"))
plt.axis("off")
plt.title("Tự chơi - Level 1")
plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_5272\3841102927.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
env3 = OrderedTreasureHuntEnv(level="level3")
result3 = build_agent("qlearning").fit(env3, episodes=20000, seed=42, **ALGO_PARAMS["qlearning"])
plots.save_episode_animation(
    env3, result3["policy"], "figures/autoplay_level3.gif",
    "Level 3 - Q-learning", fps=6, seed=42,
)
print("Đã lưu figures/autoplay_level3.gif")

Đã lưu figures/autoplay_level3.gif


## 13. Kết luận

1. **Môi trường**: bộ `gymnasium.Env` tự viết trên lưới ASCII, ba map vuông cùt kích
   thước 5x5, 7x7 và 8x8. Mô hình chuyển trạng thái là tất định, nên Value Iteration
   và Policy Iteration cho ra policy tối ưu chính xác; ba thuật toán không mô hình chỉ
   tiệm cận.
2. **Tính Markov**: `task_progress` và các biến phụ (`switch_on`, `skip_used`,
   bitmask vật phẩm) là bắt buộc. Cùng một ô với tiến độ khác nhau là hai state khác
   nhau, và điều đó được chứng minh bằng số liệu.
3. **So sánh**: trên level 1 và level 2 cả năm thuật toán đều đạt 100% thành công và
   cùng một giá trị return. Level 3 mới phân biệt được các thuật toán: Value Iteration
   và Policy Iteration tìm ra đường lấy đủ bốn vật phẩm (return 128.00), còn các thuật
   toán không mô hình rút ngắn qua ô thưởng và chỉ được 118.00 tới 120.00.
4. **Nhiều seed**: kết quả trên level 3 còn phụ thuộc vào seed. Q-learning thoát ra
   khỏi cực điểm cục bộ ở hai seed trong ba seed, còn SARSA và Monte Carlo không thoát ra seed nào.
5. **Reward**: chỉ thay đổi chi phí của một bước đã làm đổi policy tối ưu. Đó là lý do
   phải thử ít nhất hai cấu hình reward khi làm bài.
6. **Hiển thị**: môi trường được vẽ bằng matplotlib, có chế độ tự chơi chạy policy đã
   học và ghi lại thành file GIF.